In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
ayoznur_hatrec_video_dataset_path = kagglehub.dataset_download('ayoznur/hatrec-video-dataset')

print('Data source import complete.')


# V-JEPA 2 test tren HATREC

Muc tieu: lay embedding tu V-JEPA 2 (ViT-L, 300M) cho toan bo video HATREC, train 1 linear probe nhe, so Macro-F1 voi Cosmos zero-shot (18.5%) va baseline CNN+XGBoost.

Chon Runtime > Change runtime type > GPU (T4) truoc khi chay.

## 1. Cai dat thu vien

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers
!pip install -q kagglehub scikit-learn av


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 8.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 44.5 MB/s eta 0:00:00:00:0100:01


## 2. Tai dataset HATREC tu Kaggle

Dung kagglehub, khong can setup Kaggle API key thu cong neu ban dang login san trong Colab qua kagglehub.login(), hoac upload kaggle.json.

In [ ]:
import kagglehub

# Neu can dang nhap Kaggle (chi chay 1 lan, se hien link xac thuc):
# kagglehub.login()

dataset_path = kagglehub.dataset_download("ayoznur/hatrec-video-dataset")
print("Dataset path:", dataset_path)


Dataset path: /kaggle/input/datasets/ayoznur/hatrec-video-dataset


## 3. Liet ke video va gan nhan tu ten file

Ten file dang `Cycle_N_task_M.mp4`, M tu 0-6 tuong ung 7 action label.

In [ ]:
import os
import re
from glob import glob
import pandas as pd

TASK_NAMES = {
    0: "Assembling the spring",
    1: "Placing white plastic",
    2: "Screwing-1",
    3: "Inflating valve",
    4: "Placing black plastic",
    5: "Screwing-2",
    6: "Fixing cable",
}

video_files = []
for ext in ("*.mp4", "*.avi", "*.mov"):
    video_files.extend(glob(os.path.join(dataset_path, "**", ext), recursive=True))

print("Tong so video:", len(video_files))

rows = []
for path in video_files:
    fname = os.path.basename(path)
    cycle = os.path.basename(os.path.dirname(path))
    m = re.search(r"_task_(\d)", fname)
    if not m:
        continue
    label = int(m.group(1))
    rows.append({
        "path": path,
        "cycle": cycle,
        "label": label,
        "label_name": TASK_NAMES[label],
    })

df = pd.DataFrame(rows)
print("So video co nhan hop le:", len(df))
df.head()


Tong so video: 546
So video co nhan hop le: 546


,path,cycle,label,label_name
0,/kaggle/input/datasets/ayoznur/hatrec-video-da...,Cycle_69,1,Placing white plastic
1,/kaggle/input/datasets/ayoznur/hatrec-video-da...,Cycle_69,6,Fixing cable
2,/kaggle/input/datasets/ayoznur/hatrec-video-da...,Cycle_69,4,Placing black plastic
3,/kaggle/input/datasets/ayoznur/hatrec-video-da...,Cycle_69,3,Inflating valve
4,/kaggle/input/datasets/ayoznur/hatrec-video-da...,Cycle_69,0,Assembling the spring


## 4. Chia train/test theo Cycle (khong theo clip le, tranh data leakage)

In [ ]:
import random

random.seed(42)
all_cycles = sorted(df["cycle"].unique())
random.shuffle(all_cycles)

n_test = max(1, int(len(all_cycles) * 0.25))
test_cycles = set(all_cycles[:n_test])
train_cycles = set(all_cycles[n_test:])

df["split"] = df["cycle"].apply(lambda c: "test" if c in test_cycles else "train")
print(df["split"].value_counts())
print("So cycle test:", len(test_cycles), "/ so cycle train:", len(train_cycles))

# Luu split de dung lai / doi chieu sau nay
df.to_csv("hatrec_split.csv", index=False)


split
train    413
test     133
Name: count, dtype: int64
So cycle test: 19 / so cycle train: 59


## 5. Load V-JEPA 2 (ViT-L, 300M)

In [ ]:
import torch
from transformers import AutoVideoProcessor, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

hf_repo = "facebook/vjepa2-vitl-fpc64-256"
model = AutoModel.from_pretrained(hf_repo).to(device).eval()
processor = AutoVideoProcessor.from_pretrained(hf_repo)
print("Model loaded.")


Device: cuda


config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

Model loaded.


## 6. Ham decode video bang PyAV (khong dung torchcodec — tranh loi ABI/ffmpeg tung gap voi Cosmos)

In [ ]:
import av
import numpy as np

NUM_FRAMES = 64  # dung model can 64 frame

def decode_video_pyav(path, num_frames=NUM_FRAMES):
    container = av.open(path)
    stream = container.streams.video[0]
    total_frames = stream.frames
    if total_frames == 0:
        # fallback: decode het rồi đếm
        frames = [f.to_ndarray(format="rgb24") for f in container.decode(video=0)]
        total_frames = len(frames)
    else:
        frames = None

    indices = np.linspace(0, max(total_frames - 1, 0), num_frames).astype(int)

    if frames is None:
        container.seek(0)
        collected = []
        target_set = set(indices.tolist())
        for i, frame in enumerate(container.decode(video=0)):
            if i in target_set:
                collected.append((i, frame.to_ndarray(format="rgb24")))
            if len(collected) == len(target_set):
                break
        collected.sort(key=lambda x: x[0])
        frames = [f for _, f in collected]
    else:
        frames = [frames[i] for i in indices]

    # neu video ngan hon 64 frame, lap lai frame cuoi de du so luong
    while len(frames) < num_frames:
        frames.append(frames[-1])

    video = np.stack(frames[:num_frames])  # T x H x W x C
    video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
    return video

# test thu 1 video
sample_video = decode_video_pyav(df.iloc[0]["path"])
print("Sample video tensor shape:", sample_video.shape)


Sample video tensor shape: torch.Size([64, 3, 1280, 720])


## 7. Chay V-JEPA tren toan bo video, luu embedding

Chay theo batch nho de fit VRAM. Luu ket qua ra Google Drive de khong mat neu Colab bi ngat.

In [ ]:
import torch
from transformers import AutoVideoProcessor, AutoModel
import threading

hf_repo = "facebook/vjepa2-vitl-fpc64-256"

print("Số GPU khả dụng:", torch.cuda.device_count())

# Load 1 bản model riêng cho từng GPU
model_0 = AutoModel.from_pretrained(hf_repo).to("cuda:0").eval()
model_1 = AutoModel.from_pretrained(hf_repo).to("cuda:1").eval()
processor = AutoVideoProcessor.from_pretrained(hf_repo)

print("Đã load model trên cả 2 GPU.")

Số GPU khả dụng: 2


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Đã load model trên cả 2 GPU.


In [ ]:
import time

def process_subset(df_subset, model, device, results_dict, failed_list, tag):
    t0 = time.time()
    for i, row in df_subset.iterrows():
        try:
            video = decode_video_pyav(row["path"])
            inputs = processor(video, return_tensors="pt").to(device)
            with torch.no_grad():
                feat = model.get_vision_features(**inputs)
            vec = feat.mean(dim=1).squeeze(0).cpu().numpy()
            results_dict[i] = vec
        except Exception as e:
            results_dict[i] = None
            failed_list.append((row["path"], str(e)))

        if len(results_dict) % 20 == 0:
            elapsed = time.time() - t0
            print(f"[{tag}] {len(results_dict)}/{len(df_subset)} elapsed {elapsed:.1f}s")

# Chia đôi dataframe
half = len(df) // 2
df_gpu0 = df.iloc[:half]
df_gpu1 = df.iloc[half:]

embeddings_dict = {}
failed = []

t0 = time.time()
thread0 = threading.Thread(target=process_subset, args=(df_gpu0, model_0, "cuda:0", embeddings_dict, failed, "GPU0"))
thread1 = threading.Thread(target=process_subset, args=(df_gpu1, model_1, "cuda:1", embeddings_dict, failed, "GPU1"))

thread0.start()
thread1.start()
thread0.join()
thread1.join()

print(f"Xong toàn bộ trong {time.time()-t0:.1f}s. Failed: {len(failed)}")

# Gộp lại theo đúng thứ tự index gốc
embeddings = [embeddings_dict.get(i) for i in df.index]

[GPU1] 20/273 elapsed 47.0s
[GPU0] 40/273 elapsed 94.9s
[GPU1] 60/273 elapsed 141.9s
[GPU1] 80/273 elapsed 189.5s
[GPU0] 100/273 elapsed 239.7s
[GPU0] 120/273 elapsed 289.4s
[GPU1] 140/273 elapsed 338.2s
[GPU1] 160/273 elapsed 386.5s
[GPU1] 180/273 elapsed 435.6s
[GPU1] 200/273 elapsed 484.7s
[GPU0] 220/273 elapsed 535.3s
[GPU0] 240/273 elapsed 585.9s
[GPU0] 260/273 elapsed 635.7s
[GPU0] 280/273 elapsed 686.3s
[GPU1] 300/273 elapsed 736.2s
[GPU1] 320/273 elapsed 785.3s
[GPU1] 340/273 elapsed 834.4s
[GPU1] 360/273 elapsed 883.8s
[GPU1] 380/273 elapsed 933.4s
[GPU1] 400/273 elapsed 982.7s
[GPU1] 420/273 elapsed 1031.9s
[GPU1] 440/273 elapsed 1081.3s
[GPU1] 460/273 elapsed 1131.1s
[GPU0] 480/273 elapsed 1181.5s
[GPU0] 500/273 elapsed 1231.6s
[GPU0] 520/273 elapsed 1281.5s
[GPU0] 540/273 elapsed 1335.8s
Xong toàn bộ trong 1365.8s. Failed: 0


In [ ]:
import numpy as np

df["embedding_ok"] = [e is not None for e in embeddings]
valid_idx = df.index[df["embedding_ok"]].tolist()

emb_matrix = np.stack([embeddings[i] for i in valid_idx])
print("Embedding matrix shape:", emb_matrix.shape)

np.save(os.path.join(SAVE_DIR, "vjepa_embeddings.npy"), emb_matrix)
df.to_csv(os.path.join(SAVE_DIR, "hatrec_split_with_status.csv"), index=False)
print("Da luu embedding va metadata vao", SAVE_DIR)


Embedding matrix shape: (546, 1024)
Da luu embedding va metadata vao /content/vjepa_hatrec_results


## 8. Train linear probe tren embedding da dong bang

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

df_valid = df.loc[valid_idx].reset_index(drop=True)

train_mask = (df_valid["split"] == "train").values
test_mask = (df_valid["split"] == "test").values

X_train, y_train = emb_matrix[train_mask], df_valid.loc[train_mask, "label"].values
X_test, y_test = emb_matrix[test_mask], df_valid.loc[test_mask, "label"].values

print("Train size:", X_train.shape, "Test size:", X_test.shape)

probe = LogisticRegression(max_iter=2000, class_weight="balanced")
probe.fit(X_train, y_train)

y_pred = probe.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print("Macro-F1 tren test set (theo Cycle, khoa san):", round(macro_f1, 4))
print()
print(classification_report(y_test, y_pred, target_names=[TASK_NAMES[i] for i in sorted(TASK_NAMES)]))


Train size: (413, 1024) Test size: (133, 1024)
Macro-F1 tren test set (theo Cycle, khoa san): 1.0

                       precision    recall  f1-score   support

Assembling the spring       1.00      1.00      1.00        19
Placing white plastic       1.00      1.00      1.00        19
           Screwing-1       1.00      1.00      1.00        19
      Inflating valve       1.00      1.00      1.00        19
Placing black plastic       1.00      1.00      1.00        19
           Screwing-2       1.00      1.00      1.00        19
         Fixing cable       1.00      1.00      1.00        19

             accuracy                           1.00       133
            macro avg       1.00      1.00      1.00       133
         weighted avg       1.00      1.00      1.00       133



In [ ]:
def decode_single_frame_repeated(path, num_frames=64):
    video = decode_video_pyav(path, num_frames=1)  # chỉ lấy 1 frame (giữa clip)
    return video.repeat(num_frames, 1, 1, 1)

# test thử
sample_static = decode_single_frame_repeated(df.iloc[0]["path"])
print("Sample static video tensor shape:", sample_static.shape)

Sample static video tensor shape: torch.Size([64, 3, 1280, 720])


## CNN - Load CNN (ResNet18 pretrained, đóng băng) trên cả 2 GPU

In [ ]:
import torchvision.models as tv_models
import torch.nn as nn

def load_frozen_resnet(device):
    resnet = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
    resnet.fc = nn.Identity()  # bỏ lớp phân loại cuối, chỉ lấy feature 512 chiều
    resnet = resnet.to(device).eval()
    for p in resnet.parameters():
        p.requires_grad = False
    return resnet

cnn_0 = load_frozen_resnet("cuda:0")
cnn_1 = load_frozen_resnet("cuda:1")
print("Đã load ResNet18 (đóng băng) trên cả 2 GPU.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


Đã load ResNet18 (đóng băng) trên cả 2 GPU.


# Hàm trích feature bằng CNN (trung bình theo frame)

In [ ]:
import torchvision.transforms as T

cnn_transform = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_cnn_feature(video_tensor, cnn_model, device):
    # video_tensor: T x C x H x W, giá trị 0-255 uint8
    video_float = video_tensor.float() / 255.0
    video_resized = cnn_transform(video_float).to(device)
    with torch.no_grad():
        frame_feats = cnn_model(video_resized)  # T x 512
    return frame_feats.mean(dim=0).cpu().numpy()  # trung bình theo thời gian -> 512

## Chạy CNN feature extraction song song 2 GPU (video thật)

In [ ]:
def process_subset_cnn(df_subset, cnn_model, device, results_dict, failed_list, tag):
    t0 = time.time()
    for i, row in df_subset.iterrows():
        try:
            video = decode_video_pyav(row["path"])
            vec = extract_cnn_feature(video, cnn_model, device)
            results_dict[i] = vec
        except Exception as e:
            results_dict[i] = None
            failed_list.append((row["path"], str(e)))
        if len(results_dict) % 20 == 0:
            print(f"[{tag}] {len(results_dict)}/{len(df_subset)} elapsed {time.time()-t0:.1f}s")

cnn_embeddings_dict = {}
cnn_failed = []

t0 = time.time()
thread0 = threading.Thread(target=process_subset_cnn, args=(df_gpu0, cnn_0, "cuda:0", cnn_embeddings_dict, cnn_failed, "GPU0"))
thread1 = threading.Thread(target=process_subset_cnn, args=(df_gpu1, cnn_1, "cuda:1", cnn_embeddings_dict, cnn_failed, "GPU1"))
thread0.start(); thread1.start()
thread0.join(); thread1.join()

print(f"CNN xong trong {time.time()-t0:.1f}s. Failed: {len(cnn_failed)}")
cnn_embeddings = [cnn_embeddings_dict.get(i) for i in df.index]

[GPU1] 20/273 elapsed 21.1s
[GPU1] 40/273 elapsed 40.4s
[GPU1] 60/273 elapsed 60.0s
[GPU1] 80/273 elapsed 79.7s
[GPU1] 100/273 elapsed 99.3s
[GPU0] 120/273 elapsed 119.2s
[GPU0] 140/273 elapsed 138.9s
[GPU1] 160/273 elapsed 158.7s
[GPU0] 180/273 elapsed 178.4s
[GPU0] 200/273 elapsed 197.7s
[GPU0] 220/273 elapsed 217.7s
[GPU0] 240/273 elapsed 237.4s
[GPU0] 260/273 elapsed 257.4s
[GPU1] 280/273 elapsed 277.1s
[GPU1] 300/273 elapsed 296.4s
[GPU0] 320/273 elapsed 316.1s
[GPU0] 340/273 elapsed 335.3s
[GPU0] 360/273 elapsed 355.4s
[GPU0] 380/273 elapsed 374.6s
[GPU0] 400/273 elapsed 393.5s
[GPU0] 420/273 elapsed 413.4s
[GPU0] 440/273 elapsed 434.3s
[GPU0] 460/273 elapsed 454.1s
[GPU0] 480/273 elapsed 474.3s
[GPU0] 500/273 elapsed 494.5s
[GPU0] 520/273 elapsed 514.0s
[GPU1] 540/273 elapsed 533.8s
CNN xong trong 539.5s. Failed: 0


## Train XGBoost trên CNN feature

In [ ]:
!pip install -q xgboost
from xgboost import XGBClassifier

df["cnn_embedding_ok"] = [e is not None for e in cnn_embeddings]
valid_idx_cnn = df.index[df["cnn_embedding_ok"]].tolist()

cnn_matrix = np.stack([cnn_embeddings[i] for i in valid_idx_cnn])
df_valid_cnn = df.loc[valid_idx_cnn].reset_index(drop=True)

train_mask_cnn = (df_valid_cnn["split"] == "train").values
test_mask_cnn = (df_valid_cnn["split"] == "test").values

X_train_cnn = cnn_matrix[train_mask_cnn]
y_train_cnn = df_valid_cnn.loc[train_mask_cnn, "label"].values
X_test_cnn = cnn_matrix[test_mask_cnn]
y_test_cnn = df_valid_cnn.loc[test_mask_cnn, "label"].values

xgb_model = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="mlogloss")
xgb_model.fit(X_train_cnn, y_train_cnn)

y_pred_cnn = xgb_model.predict(X_test_cnn)
macro_f1_cnn = f1_score(y_test_cnn, y_pred_cnn, average="macro")

print("Macro-F1 CNN + XGBoost (video thật):", round(macro_f1_cnn, 4))
print(classification_report(y_test_cnn, y_pred_cnn, target_names=[TASK_NAMES[i] for i in sorted(TASK_NAMES)]))

Macro-F1 CNN + XGBoost (video thật): 1.0
                       precision    recall  f1-score   support

Assembling the spring       1.00      1.00      1.00        19
Placing white plastic       1.00      1.00      1.00        19
           Screwing-1       1.00      1.00      1.00        19
      Inflating valve       1.00      1.00      1.00        19
Placing black plastic       1.00      1.00      1.00        19
           Screwing-2       1.00      1.00      1.00        19
         Fixing cable       1.00      1.00      1.00        19

             accuracy                           1.00       133
            macro avg       1.00      1.00      1.00       133
         weighted avg       1.00      1.00      1.00       133



## 1 frame lặp lại CNN

In [ ]:
def process_subset_cnn_static(df_subset, cnn_model, device, results_dict, failed_list, tag):
    t0 = time.time()
    for i, row in df_subset.iterrows():
        try:
            video = decode_single_frame_repeated(row["path"])
            vec = extract_cnn_feature(video, cnn_model, device)
            results_dict[i] = vec
        except Exception as e:
            results_dict[i] = None
            failed_list.append((row["path"], str(e)))
        if len(results_dict) % 20 == 0:
            print(f"[{tag}] {len(results_dict)}/{len(df_subset)} elapsed {time.time()-t0:.1f}s")

cnn_static_dict = {}
cnn_static_failed = []
t0 = time.time()
thread0 = threading.Thread(target=process_subset_cnn_static, args=(df_gpu0, cnn_0, "cuda:0", cnn_static_dict, cnn_static_failed, "GPU0"))
thread1 = threading.Thread(target=process_subset_cnn_static, args=(df_gpu1, cnn_1, "cuda:1", cnn_static_dict, cnn_static_failed, "GPU1"))
thread0.start(); thread1.start()
thread0.join(); thread1.join()

cnn_static_embeddings = [cnn_static_dict.get(i) for i in df.index]
cnn_static_matrix = np.stack([cnn_static_embeddings[i] for i in valid_idx_cnn])

X_train_cnn_s = cnn_static_matrix[train_mask_cnn]
X_test_cnn_s = cnn_static_matrix[test_mask_cnn]

xgb_static = XGBClassifier(n_estimators=200, max_depth=4, eval_metric="mlogloss")
xgb_static.fit(X_train_cnn_s, y_train_cnn)
y_pred_cnn_s = xgb_static.predict(X_test_cnn_s)
macro_f1_cnn_static = f1_score(y_test_cnn, y_pred_cnn_s, average="macro")

print("Macro-F1 CNN + XGBoost (1 frame tĩnh):", round(macro_f1_cnn_static, 4))

[GPU1] 20/273 elapsed 17.8s
[GPU0] 40/273 elapsed 35.7s
[GPU1] 60/273 elapsed 53.5s
[GPU0] 80/273 elapsed 71.4s
[GPU1] 100/273 elapsed 89.4s
[GPU1] 120/273 elapsed 107.3s
[GPU1] 140/273 elapsed 125.2s
[GPU0] 160/273 elapsed 143.0s
[GPU1] 180/273 elapsed 161.0s
[GPU0] 200/273 elapsed 178.9s
[GPU0] 220/273 elapsed 196.9s
[GPU1] 240/273 elapsed 214.9s
[GPU0] 260/273 elapsed 232.7s
[GPU1] 280/273 elapsed 250.8s
[GPU0] 300/273 elapsed 268.7s
[GPU1] 320/273 elapsed 286.6s
[GPU0] 340/273 elapsed 304.6s
[GPU1] 360/273 elapsed 322.5s
[GPU0] 380/273 elapsed 340.5s
[GPU1] 400/273 elapsed 358.5s
[GPU1] 420/273 elapsed 376.5s
[GPU1] 440/273 elapsed 394.5s
[GPU0] 460/273 elapsed 412.5s
[GPU0] 480/273 elapsed 430.5s
[GPU0] 500/273 elapsed 448.4s
[GPU0] 520/273 elapsed 466.3s
[GPU0] 540/273 elapsed 484.3s
Macro-F1 CNN + XGBoost (1 frame tĩnh): 0.9552


## Test single frame với V-Jepa

In [ ]:
def process_subset_static(df_subset, model, device, results_dict, failed_list, tag):
    t0 = time.time()
    for i, row in df_subset.iterrows():
        try:
            video = decode_single_frame_repeated(row["path"])  # KHÁC: dùng static thay vì decode_video_pyav
            inputs = processor(video, return_tensors="pt").to(device)
            with torch.no_grad():
                feat = model.get_vision_features(**inputs)
            vec = feat.mean(dim=1).squeeze(0).cpu().numpy()
            results_dict[i] = vec
        except Exception as e:
            results_dict[i] = None
            failed_list.append((row["path"], str(e)))

        if len(results_dict) % 20 == 0:
            elapsed = time.time() - t0
            print(f"[{tag}] {len(results_dict)}/{len(df_subset)} elapsed {elapsed:.1f}s")

embeddings_dict_static = {}
failed_static = []

t0 = time.time()
thread0 = threading.Thread(target=process_subset_static, args=(df_gpu0, model_0, "cuda:0", embeddings_dict_static, failed_static, "GPU0"))
thread1 = threading.Thread(target=process_subset_static, args=(df_gpu1, model_1, "cuda:1", embeddings_dict_static, failed_static, "GPU1"))

thread0.start()
thread1.start()
thread0.join()
thread1.join()

print(f"Xong toàn bộ (static) trong {time.time()-t0:.1f}s. Failed: {len(failed_static)}")

embeddings_static = [embeddings_dict_static.get(i) for i in df.index]

[GPU0] 20/273 elapsed 50.3s
[GPU1] 40/273 elapsed 97.6s
[GPU1] 60/273 elapsed 144.9s
[GPU0] 80/273 elapsed 192.3s
[GPU0] 100/273 elapsed 241.4s
[GPU0] 120/273 elapsed 290.4s
[GPU1] 140/273 elapsed 338.4s
[GPU1] 160/273 elapsed 385.6s
[GPU1] 180/273 elapsed 433.0s
[GPU0] 200/273 elapsed 481.7s
[GPU0] 220/273 elapsed 530.8s
[GPU1] 240/273 elapsed 579.9s
[GPU1] 260/273 elapsed 627.3s
[GPU1] 280/273 elapsed 674.8s
[GPU1] 300/273 elapsed 722.4s
[GPU0] 320/273 elapsed 771.1s
[GPU0] 340/273 elapsed 820.0s
[GPU0] 360/273 elapsed 868.7s
[GPU1] 380/273 elapsed 917.5s
[GPU1] 400/273 elapsed 965.4s
[GPU1] 420/273 elapsed 1012.9s
[GPU1] 440/273 elapsed 1060.5s
[GPU0] 460/273 elapsed 1109.4s
[GPU0] 480/273 elapsed 1158.7s
[GPU0] 500/273 elapsed 1207.6s
[GPU1] 520/273 elapsed 1255.6s
[GPU0] 540/273 elapsed 1310.0s
Xong toàn bộ (static) trong 1338.9s. Failed: 0


In [ ]:
df["embedding_static_ok"] = [e is not None for e in embeddings_static]
valid_idx_static = df.index[df["embedding_static_ok"]].tolist()

emb_matrix_static = np.stack([embeddings_static[i] for i in valid_idx_static])
df_valid_static = df.loc[valid_idx_static].reset_index(drop=True)

train_mask_static = (df_valid_static["split"] == "train").values
test_mask_static = (df_valid_static["split"] == "test").values

X_train_static = emb_matrix_static[train_mask_static]
y_train_static = df_valid_static.loc[train_mask_static, "label"].values
X_test_static = emb_matrix_static[test_mask_static]
y_test_static = df_valid_static.loc[test_mask_static, "label"].values

probe_static = LogisticRegression(max_iter=2000, class_weight="balanced")
probe_static.fit(X_train_static, y_train_static)

y_pred_static = probe_static.predict(X_test_static)
macro_f1_static = f1_score(y_test_static, y_pred_static, average="macro")

print("=== SO SÁNH ===")
print(f"Macro-F1 với video thật (64 frame chuyển động): {macro_f1:.4f}")
print(f"Macro-F1 với 1 frame tĩnh lặp lại 64 lần:        {macro_f1_static:.4f}")
print()
if macro_f1_static > 0.9:
    print("=> XÁC NHẬN: model đang phân loại dựa trên vật thể/màu sắc tĩnh, KHÔNG cần chuyển động thật.")
    print("   Kết quả Macro-F1 = 1.0 trước đó KHÔNG phản ánh khả năng hiểu hành động qua thời gian.")
else:
    print("=> Chuyển động thật đóng vai trò quan trọng — model cần xem video thật để phân loại đúng.")

print()
print(classification_report(y_test_static, y_pred_static, target_names=[TASK_NAMES[i] for i in sorted(TASK_NAMES)]))

=== SO SÁNH ===
Macro-F1 với video thật (64 frame chuyển động): 1.0000
Macro-F1 với 1 frame tĩnh lặp lại 64 lần:        0.9851

=> XÁC NHẬN: model đang phân loại dựa trên vật thể/màu sắc tĩnh, KHÔNG cần chuyển động thật.
   Kết quả Macro-F1 = 1.0 trước đó KHÔNG phản ánh khả năng hiểu hành động qua thời gian.

                       precision    recall  f1-score   support

Assembling the spring       1.00      1.00      1.00        19
Placing white plastic       1.00      0.95      0.97        19
           Screwing-1       1.00      1.00      1.00        19
      Inflating valve       1.00      1.00      1.00        19
Placing black plastic       0.90      1.00      0.95        19
           Screwing-2       1.00      0.95      0.97        19
         Fixing cable       1.00      1.00      1.00        19

             accuracy                           0.98       133
            macro avg       0.99      0.98      0.99       133
         weighted avg       0.99      0.98      0.99     